# Claussifier - Threshold Optimization

**Goal:** Find optimal classification thresholds for each risk category to improve precision

**What we'll accomplish:**
- Load trained BERT model from Google Drive
- Test different thresholds (0.3 - 0.7) on validation set
- Find optimal threshold per class
- Evaluate on test set with optimized thresholds
- Compare with baseline (threshold = 0.5)

---

## 1. Setup & Installation

In [ ]:
# Install required libraries
!pip install -q datasets transformers torch scikit-learn

In [ ]:
# Import libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("✓ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive & Load Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set up directories
PROJECT_DIR = "/content/drive/MyDrive/Claussifier"
MODEL_DIR = f"{PROJECT_DIR}/models/legalbert_final_model"
RESULTS_DIR = f"{PROJECT_DIR}/results"

print("✓ Google Drive mounted!")

In [ ]:
# Load configuration
LABEL_NAMES = [
    "Limitation of liability",
    "Unilateral termination",
    "Unilateral change",
    "Content removal",
    "Contract by using",
    "Choice of law",
    "Jurisdiction",
    "Arbitration"
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load trained model and tokenizer
print(f"Loading model from: {MODEL_DIR}")

model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

print("✓ Model and tokenizer loaded successfully!")

## 3. Load Dataset

In [ ]:
# Load dataset
print("Loading LexGLUE unfair_tos dataset...")
dataset = load_dataset("lex_glue", "unfair_tos")

print("\n✓ Dataset loaded successfully!")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"Test: {len(dataset['test'])} examples")

In [ ]:
# Custom Dataset class (same as training)
class ClauseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label_list = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        label_tensor = torch.zeros(8, dtype=torch.float)
        for label_idx in label_list:
            label_tensor[label_idx] = 1.0
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': label_tensor
        }

# Create datasets
val_dataset = ClauseDataset(
    dataset['validation']['text'],
    dataset['validation']['labels'],
    tokenizer
)

test_dataset = ClauseDataset(
    dataset['test']['text'],
    dataset['test']['labels'],
    tokenizer
)

# Create DataLoaders
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("✓ Datasets and DataLoaders created!")

## 4. Get Model Predictions (Probabilities)

In [ ]:
def get_predictions(model, dataloader, device):
    """Get raw probabilities and true labels."""
    model.eval()
    
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Getting predictions"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.sigmoid(logits)
            
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    
    return all_probs, all_labels

print("✓ Prediction function defined!")

In [ ]:
# Get predictions on validation set
print("Getting predictions on validation set...")
val_probs, val_labels = get_predictions(model, val_loader, device)

print(f"\n✓ Validation predictions obtained!")
print(f"Shape: {val_probs.shape}")
print(f"Probability range: [{val_probs.min():.3f}, {val_probs.max():.3f}]")

## 5. Threshold Grid Search

In [ ]:
# Define thresholds to test
THRESHOLDS = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

print(f"Testing {len(THRESHOLDS)} thresholds: {THRESHOLDS}")
print(f"For {len(LABEL_NAMES)} classes")
print(f"Total combinations: {len(THRESHOLDS) * len(LABEL_NAMES)}")

In [ ]:
# Grid search for optimal thresholds
results = []

print("\nRunning grid search...\n")

for class_idx, class_name in enumerate(LABEL_NAMES):
    print(f"Class {class_idx + 1}/8: {class_name}")
    
    class_probs = val_probs[:, class_idx]
    class_labels = val_labels[:, class_idx]
    
    best_f1 = 0
    best_threshold = 0.5
    
    for threshold in THRESHOLDS:
        preds = (class_probs >= threshold).astype(int)
        
        precision = precision_score(class_labels, preds, zero_division=0)
        recall = recall_score(class_labels, preds, zero_division=0)
        f1 = f1_score(class_labels, preds, zero_division=0)
        
        results.append({
            'class_idx': class_idx,
            'class_name': class_name,
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1
        })
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    print(f"  Best threshold: {best_threshold} (F1: {best_f1:.4f})")

results_df = pd.DataFrame(results)
print("\n✓ Grid search complete!")

## 6. Find Optimal Thresholds

In [ ]:
# Find best threshold for each class
optimal_thresholds = []

print("\nOptimal Thresholds per Class:\n")
print("="*80)

for class_idx, class_name in enumerate(LABEL_NAMES):
    class_results = results_df[results_df['class_idx'] == class_idx]
    best_row = class_results.loc[class_results['f1'].idxmax()]
    
    optimal_thresholds.append(best_row['threshold'])
    
    print(f"{class_idx + 1}. {class_name}")
    print(f"   Optimal Threshold: {best_row['threshold']}")
    print(f"   F1: {best_row['f1']:.4f} | Precision: {best_row['precision']:.4f} | Recall: {best_row['recall']:.4f}")
    print()

print("="*80)
print(f"\nOptimal thresholds: {optimal_thresholds}")

## 7. Visualize Threshold Impact

In [ ]:
# Plot F1 vs Threshold for each class
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for class_idx, class_name in enumerate(LABEL_NAMES):
    class_results = results_df[results_df['class_idx'] == class_idx]
    
    ax = axes[class_idx]
    ax.plot(class_results['threshold'], class_results['precision'], 'b-', label='Precision', marker='o')
    ax.plot(class_results['threshold'], class_results['recall'], 'r-', label='Recall', marker='s')
    ax.plot(class_results['threshold'], class_results['f1'], 'g-', label='F1', marker='^', linewidth=2)
    
    # Mark optimal threshold
    optimal_thresh = optimal_thresholds[class_idx]
    ax.axvline(x=optimal_thresh, color='purple', linestyle='--', alpha=0.7, label=f'Optimal ({optimal_thresh})')
    
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title(class_name, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/threshold_optimization_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Threshold curves saved!")

## 8. Evaluate on Test Set with Optimal Thresholds

In [ ]:
# Get predictions on test set
print("Getting predictions on test set...")
test_probs, test_labels = get_predictions(model, test_loader, device)

print("✓ Test predictions obtained!")

In [ ]:
def evaluate_with_thresholds(probs, labels, thresholds):
    """Evaluate with custom thresholds per class."""
    preds = np.zeros_like(probs)
    
    for class_idx, threshold in enumerate(thresholds):
        preds[:, class_idx] = (probs[:, class_idx] >= threshold).astype(int)
    
    # Compute metrics
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    macro_precision = precision_score(labels, preds, average='macro', zero_division=0)
    macro_recall = recall_score(labels, preds, average='macro', zero_division=0)
    
    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    micro_precision = precision_score(labels, preds, average='micro', zero_division=0)
    micro_recall = recall_score(labels, preds, average='micro', zero_division=0)
    
    per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
    per_class_precision = precision_score(labels, preds, average=None, zero_division=0)
    per_class_recall = recall_score(labels, preds, average=None, zero_division=0)
    
    return {
        'macro_f1': macro_f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'micro_f1': micro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'per_class_f1': per_class_f1,
        'per_class_precision': per_class_precision,
        'per_class_recall': per_class_recall
    }

print("✓ Evaluation function defined!")

In [ ]:
# Baseline: threshold = 0.5 for all classes
baseline_thresholds = [0.5] * 8
baseline_metrics = evaluate_with_thresholds(test_probs, test_labels, baseline_thresholds)

print("BASELINE (Threshold = 0.5 for all classes)")
print("="*80)
print(f"Macro F1:        {baseline_metrics['macro_f1']:.4f}")
print(f"Macro Precision: {baseline_metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {baseline_metrics['macro_recall']:.4f}")
print(f"Micro F1:        {baseline_metrics['micro_f1']:.4f}")
print("="*80)

In [ ]:
# Optimized: use optimal thresholds
optimized_metrics = evaluate_with_thresholds(test_probs, test_labels, optimal_thresholds)

print("\nOPTIMIZED (Optimal thresholds per class)")
print("="*80)
print(f"Macro F1:        {optimized_metrics['macro_f1']:.4f}")
print(f"Macro Precision: {optimized_metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {optimized_metrics['macro_recall']:.4f}")
print(f"Micro F1:        {optimized_metrics['micro_f1']:.4f}")
print("="*80)

## 9. Compare Results

In [ ]:
# Comparison table
comparison_df = pd.DataFrame({
    'Metric': ['Macro F1', 'Macro Precision', 'Macro Recall', 'Micro F1'],
    'Baseline (0.5)': [
        baseline_metrics['macro_f1'],
        baseline_metrics['macro_precision'],
        baseline_metrics['macro_recall'],
        baseline_metrics['micro_f1']
    ],
    'Optimized': [
        optimized_metrics['macro_f1'],
        optimized_metrics['macro_precision'],
        optimized_metrics['macro_recall'],
        optimized_metrics['micro_f1']
    ]
})

comparison_df['Improvement'] = comparison_df['Optimized'] - comparison_df['Baseline (0.5)']
comparison_df['Improvement %'] = (comparison_df['Improvement'] / comparison_df['Baseline (0.5)']) * 100

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Save comparison
comparison_df.to_csv(f'{RESULTS_DIR}/threshold_optimization_comparison.csv', index=False)
print("\n✓ Comparison saved to CSV")

In [ ]:
# Per-class comparison
per_class_comparison = pd.DataFrame({
    'Risk Category': LABEL_NAMES,
    'Optimal Threshold': optimal_thresholds,
    'Baseline F1': baseline_metrics['per_class_f1'],
    'Optimized F1': optimized_metrics['per_class_f1'],
    'F1 Improvement': optimized_metrics['per_class_f1'] - baseline_metrics['per_class_f1'],
    'Baseline Precision': baseline_metrics['per_class_precision'],
    'Optimized Precision': optimized_metrics['per_class_precision'],
    'Baseline Recall': baseline_metrics['per_class_recall'],
    'Optimized Recall': optimized_metrics['per_class_recall']
})

print("\n" + "="*80)
print("PER-CLASS COMPARISON")
print("="*80)
print(per_class_comparison.to_string(index=False))
print("="*80)

# Save per-class comparison
per_class_comparison.to_csv(f'{RESULTS_DIR}/threshold_optimization_per_class.csv', index=False)
print("\n✓ Per-class comparison saved to CSV")

## 10. Visualize Improvements

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall metrics
metrics = ['Macro F1', 'Macro Precision', 'Macro Recall', 'Micro F1']
baseline_vals = [
    baseline_metrics['macro_f1'],
    baseline_metrics['macro_precision'],
    baseline_metrics['macro_recall'],
    baseline_metrics['micro_f1']
]
optimized_vals = [
    optimized_metrics['macro_f1'],
    optimized_metrics['macro_precision'],
    optimized_metrics['macro_recall'],
    optimized_metrics['micro_f1']
]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, baseline_vals, width, label='Baseline (0.5)', alpha=0.8)
axes[0].bar(x + width/2, optimized_vals, width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Metric', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Overall Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, 1.0)

# Per-class F1
x2 = np.arange(len(LABEL_NAMES))
axes[1].bar(x2 - width/2, baseline_metrics['per_class_f1'], width, label='Baseline (0.5)', alpha=0.8)
axes[1].bar(x2 + width/2, optimized_metrics['per_class_f1'], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Risk Category', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('Per-Class F1 Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(LABEL_NAMES, rotation=45, ha='right')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/threshold_optimization_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Comparison charts saved!")

## 11. Save Optimal Thresholds

In [ ]:
# Save optimal thresholds to JSON
threshold_config = {
    'optimal_thresholds': optimal_thresholds,
    'label_names': LABEL_NAMES,
    'baseline_metrics': {
        'macro_f1': float(baseline_metrics['macro_f1']),
        'macro_precision': float(baseline_metrics['macro_precision']),
        'macro_recall': float(baseline_metrics['macro_recall'])
    },
    'optimized_metrics': {
        'macro_f1': float(optimized_metrics['macro_f1']),
        'macro_precision': float(optimized_metrics['macro_precision']),
        'macro_recall': float(optimized_metrics['macro_recall'])
    },
    'improvement': {
        'macro_f1': float(optimized_metrics['macro_f1'] - baseline_metrics['macro_f1']),
        'macro_precision': float(optimized_metrics['macro_precision'] - baseline_metrics['macro_precision']),
        'macro_recall': float(optimized_metrics['macro_recall'] - baseline_metrics['macro_recall'])
    }
}

with open(f'{RESULTS_DIR}/optimal_thresholds.json', 'w') as f:
    json.dump(threshold_config, f, indent=2)

print("✓ Optimal thresholds saved to JSON!")
print(f"\nLocation: {RESULTS_DIR}/optimal_thresholds.json")

## 12. Final Summary

In [ ]:
# Generate summary report
summary = f"""
{'='*80}
THRESHOLD OPTIMIZATION SUMMARY
{'='*80}

BASELINE PERFORMANCE (Threshold = 0.5 for all classes)
  Macro F1:        {baseline_metrics['macro_f1']:.4f}
  Macro Precision: {baseline_metrics['macro_precision']:.4f}
  Macro Recall:    {baseline_metrics['macro_recall']:.4f}

OPTIMIZED PERFORMANCE (Class-specific thresholds)
  Macro F1:        {optimized_metrics['macro_f1']:.4f}
  Macro Precision: {optimized_metrics['macro_precision']:.4f}
  Macro Recall:    {optimized_metrics['macro_recall']:.4f}

IMPROVEMENTS
  Macro F1:        {optimized_metrics['macro_f1'] - baseline_metrics['macro_f1']:+.4f} ({((optimized_metrics['macro_f1'] - baseline_metrics['macro_f1']) / baseline_metrics['macro_f1'] * 100):+.2f}%)
  Macro Precision: {optimized_metrics['macro_precision'] - baseline_metrics['macro_precision']:+.4f} ({((optimized_metrics['macro_precision'] - baseline_metrics['macro_precision']) / baseline_metrics['macro_precision'] * 100):+.2f}%)
  Macro Recall:    {optimized_metrics['macro_recall'] - baseline_metrics['macro_recall']:+.4f} ({((optimized_metrics['macro_recall'] - baseline_metrics['macro_recall']) / baseline_metrics['macro_recall'] * 100):+.2f}%)

OPTIMAL THRESHOLDS
"""

for i, (name, thresh) in enumerate(zip(LABEL_NAMES, optimal_thresholds)):
    summary += f"  {i+1}. {name}: {thresh}\n"

summary += f"""
{'='*80}
FILES GENERATED
  - {RESULTS_DIR}/threshold_optimization_curves.png
  - {RESULTS_DIR}/threshold_optimization_comparison.png
  - {RESULTS_DIR}/threshold_optimization_comparison.csv
  - {RESULTS_DIR}/threshold_optimization_per_class.csv
  - {RESULTS_DIR}/optimal_thresholds.json
{'='*80}
"""

print(summary)

# Save summary
with open(f'{RESULTS_DIR}/threshold_optimization_summary.txt', 'w') as f:
    f.write(summary)

print("\n✓ Summary report saved!")

---
**All files saved to Google Drive:**
- `/MyDrive/Claussifier/results/optimal_thresholds.json`
- `/MyDrive/Claussifier/results/threshold_optimization_*.png`
- `/MyDrive/Claussifier/results/threshold_optimization_*.csv`